# 49장 · 검색·인용 품질과 비용 측정 조건

실패 지점을 검색과 인용으로 나누면 고칠 위치가 달라집니다. 비용을 기록할 때도 실제로 호출하지 않은 값과 아직 재지 않은 값을 구분합니다.

Q03의 기준선과 개선 후보를 비교해 검색 실패와 인용 실패를 구분합니다. 모델 호출 수와 측정하지 않은 비용·지연도 별도 필드로 남깁니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요.


## 실패 단계별 평가

완성 품질 기록｜기준선에서는 p3을 찾지 못해 답을 만들지 않았습니다. 보류 후보에서는 p3을 찾아 생성 규칙까지 갔지만 엉뚱한 p2를 인용해 응답을 막았습니다. 팀의 우선 수정 항목 I49-Q03은 후보의 생성 규칙입니다.


### 셀 01 · 검색 실패와 인용 실패 판별

각 행은 실제 외부 모델 없이 제공 예제를 돌렸던 로컬 기록입니다.

이번 실험에서 고정할 것은 p3을 찾는 error_keyword 후보의 검색입니다. 현재 서비스는 여전히 baseline을 씁니다. 생성 방식을 바꿔 비교할 때는 두 실험 모두 같은 p3을 받았는지 먼저 확인하세요.


In [1]:
original = {'retrieved_ids': [], 'generation_calls': 0, 'status': 'no_evidence'}
candidate = {'retrieved_ids': ['course-guide-v1:p3'],
 'generation_calls': 1,
 'status': 'invalid_citation'}
print(original)
print(candidate)

{'retrieved_ids': [], 'generation_calls': 0, 'status': 'no_evidence'}
{'retrieved_ids': ['course-guide-v1:p3'], 'generation_calls': 1, 'status': 'invalid_citation'}


기준선의 생성 호출 0은 근거를 찾지 못해 건너뛰었다는 뜻입니다. 후보의 1은 고정 생성 함수를 한 번 실행했다는 뜻입니다. Q03은 이미 공개된 문항이므로 진단 증거로만 사용합니다.


## 호출 횟수와 미측정 비용

실험 기록 예시｜실행 모드는 로컬 규칙, 외부 모델 호출은 0회, 모델 비용과 모델 지연은 미측정입니다. 로컬 코드의 실행 시간에는 화면 대기와 실제 모델 응답시간이 포함되지 않습니다.


### 셀 02 · 미측정 비용과 호출 수 구분

후보의 비용 관측 필드만 꺼내 봅니다.


In [2]:
usage = {'external_model_calls': 0, 'model_cost': None, 'model_latency': None}
print(usage)

{'external_model_calls': 0, 'model_cost': None, 'model_latency': None}


외부 호출 수는 0이고 비용과 지연은 None입니다. 측정하지 않은 값을 0원이나 0초로 덮어쓰면 실제 모델과 비교할 수 없습니다. 팀이 실제 연동을 사용한다면 요청별 입력량·출력량·적용 요금·측정 구간을 남겨 계산하세요.

external_model_calls의 0은 제공된 실행 기록에 적힌 외부 호출 횟수입니다. 반면 model_latency의 None에는 측정값이 없습니다. 이를 0초로 바꿔 평균에 포함하면 실제 모델을 호출한 실행과 비교할 때 지연을 과소평가하게 됩니다.


#### 호출 수 0과 비용 미측정

<mark>외부 호출 횟수와 과금액은 서로 다른 관찰값입니다.</mark> 로컬 기준선에서 세지 않은 비용을 0원으로 채우면 유료 실행과의 차이를 계산할 근거가 사라집니다. 팀 기록에는 횟수와 금액의 측정 여부를 따로 적으세요.

선수 개념 · 39장 RAG 개선안의 품질과 비용 비교

### 0회와 측정값 없음 구별하기

external_model_calls의 0은 외부 모델을 부르지 않았다는 횟수입니다. model_cost와 model_latency의 None은 관측한 값이 없다는 뜻입니다. 평균을 낼 때 None을 0원이나 0초로 넣으면 실제보다 낮은 수치가 됩니다.


## 실패한 한 요청의 검색과 인용을 따로 채점하기

Q03의 기대 근거는 p3입니다. 후보에서 p3을 찾아오고도 잘못된 p2를 인용한 경우를 검색 성공 한 점으로만 보고하면 팀이 잘못된 코드를 고치게 됩니다.

### 셀 03 · 후보와 인용을 같은 질문에서 따로 검사

46장에서 기록한 Q03 후보의 검색 목록을 사용합니다. 아래 p2는 생성기가 잘못 붙인 인용으로 넣은 값입니다.

In [3]:
def score_trace(expected_id, retrieved_ids, cited_ids):
    retrieved = expected_id in retrieved_ids
    backed = bool(cited_ids) and set(cited_ids).issubset(retrieved_ids)
    answered_with_expected = expected_id in cited_ids and backed
    return retrieved, answered_with_expected

expected = "course-guide-v1:p3"
print(score_trace(expected, candidate["retrieved_ids"], ["course-guide-v1:p2"]))

(True, False)


True는 p3을 찾았다는 뜻이고 False는 기대 문단을 근거로 답하지 못했다는 뜻입니다. 검색 후보 수를 늘려도 언제나 p2를 붙이는 생성 규칙은 그대로 남습니다.

### 수정 이슈 I49-Q03

<table><thead><tr><th scope="col">재현 입력</th><th scope="col">최초로 어긋난 값</th><th scope="col">수정 뒤 다시 볼 값</th></tr></thead><tbody><tr><td>Q03 제출 오류</td><td>검색 p3, 생성 인용 p2</td><td>인용 ID가 선택된 원문 p3과 같은지</td></tr><tr><td>기존 성공 Q01</td><td>기준선의 검색과 인용 모두 p2</td><td>같은 질문에서 p2 인용이 유지되는지</td></tr></tbody></table>

실패 항목이 하나라도 실제 원문을 읽고 답변 내용이 그 문단 안에서 성립하는지 별도로 확인합니다. ID가 일치해도 문장을 틀리게 요약할 수 있습니다.

<mark>검색 적중과 근거에 맞춘 답변은 따로 확인해야 합니다.</mark> 찾은 문서와 인용한 문서를 기록해 두면 답변 생성 코드의 어느 부분을 고칠지 좁힐 수 있습니다.

50장에는 Q03의 실패 입력과 더불어 고치기 전부터 성공한 Q01도 회귀 질문으로 전달합니다.

## 돌아보기

다음 단계에 넘길 한 줄은 ‘검색 고정, Q03에서 생성 인용만 수정, 나머지 세 질문으로 회귀 확인’입니다. 팀에서 실패가 가장 많은 원인이 다르면 근거 행과 함께 우선순위를 바꾸세요.
